In [15]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.model_selection import train_test_split

In [2]:
os.chdir('/home/ntdung/Medical')

In [3]:
excel_path = 'data/participants.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66.0,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69.0,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23.0,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54.0,control,f,sub-BrainAge023212,RocklandSample


In [4]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4242
Samples with decimal age values: 706


In [5]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54,control,f,sub-BrainAge023212,RocklandSample


In [6]:
def load_middle_slices(nii_path):
    img = nib.load(nii_path).get_fdata()

    x, y, z = img.shape
    axial_slice = img[:, :, z // 2]
    sagittal_slice = img[x // 2, :, :]
    coronal_slice = img[:, y // 2, :]

    # Normalize
    axial_slice = (axial_slice - axial_slice.min()) / (axial_slice.max() - axial_slice.min())
    sagittal_slice = (sagittal_slice - sagittal_slice.min()) / (sagittal_slice.max() - sagittal_slice.min())
    coronal_slice = (coronal_slice - coronal_slice.min()) / (coronal_slice.max() - coronal_slice.min())

    return axial_slice, sagittal_slice, coronal_slice

In [7]:
class BrainMRIDataset(Dataset):
    def __init__(self, dataframe, root_dir):
        self.df = dataframe.reset_index(drop=True)
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subject_id = row['subject_id']
        age = row['subject_age']
        sex = 0 if row['subject_sex'] == 'm' else 1

        nii_path = os.path.join(self.root_dir, subject_id, 'anat', f'{subject_id}_T1w.nii.gz')
        axial, sagittal, coronal = load_middle_slices(nii_path)

        # Stack to tensor (3, H, W)
        image = np.stack([axial, sagittal, coronal], axis=0).astype(np.float32)

        while True:
            cf_idx = random.randint(0, len(self.df) - 1)
            if cf_idx != idx:
                break
        cf_row = self.df.iloc[cf_idx]
        cf_age = cf_row['subject_age']
        cf_sex = 0 if cf_row['subject_sex'] == 'm' else 1

        return {
            'image': torch.tensor(image),
            'age': torch.tensor(age, dtype=torch.float32),
            'sex': torch.tensor(sex, dtype=torch.long),
            'cf_age': torch.tensor(cf_age, dtype=torch.float32),
            'cf_sex': torch.tensor(cf_sex, dtype=torch.long),
            'subject_id': subject_id
        }


In [8]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = BrainMRIDataset(train_df, 'data')
test_dataset = BrainMRIDataset(test_df, 'data')

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [9]:
print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

Number of batches in train_loader: 1979
Number of batches in test_loader: 495


In [10]:
first_batch = next(iter(train_loader))
print("Image shape:", first_batch['image'].shape)  # (B, 3, H, W)
print("Age:", first_batch['age'])
print("Sex:", first_batch['sex'])
print("CF Age:", first_batch['cf_age'])
print("CF Sex:", first_batch['cf_sex'])
print("Subject IDs:", first_batch['subject_id'])

Image shape: torch.Size([2, 3, 130, 130])
Age: tensor([70., 18.])
Sex: tensor([1, 1])
CF Age: tensor([20., 75.])
CF Sex: tensor([1, 1])
Subject IDs: ['sub-BrainAge020804', 'sub-BrainAge010928']


In [11]:
# Age normalization
def normalize_age(age, min_age=18, max_age=97):
    return (age - min_age) / (max_age - min_age)

# Get example batch
batch = first_batch

x = batch['image']              # (B, 3, 130, 130)
cf_age = batch['cf_age']        # (B,)
cf_sex = batch['cf_sex']        # (B,)

# Normalize age
cf_age_norm = normalize_age(cf_age)  # (B,)

# Create condition vector (B, 2)
c_vec = torch.stack([cf_age_norm, cf_sex.float()], dim=1)

# Expand to (B, 2, 130, 130)
B, _, H, W = x.shape
c_map = c_vec.unsqueeze(-1).unsqueeze(-1).repeat(1, 1, H, W)

# Concatenate condition with input image => (B, 5, 130, 130)
x_input = torch.cat([x, c_map], dim=1)

print("Input to generator:", x_input.shape)

Input to generator: torch.Size([2, 5, 130, 130])


In [16]:
def default_unet_features():
    return [[16, 32, 32, 32], [32, 32, 32, 32, 32, 16, 16]]

class ConvBlock(nn.Module):
    def __init__(self, ndims, in_channels, out_channels, stride):
        super().__init__()
        conv_fn = {1: nn.Conv1d, 2: nn.Conv2d, 3: nn.Conv3d}[ndims]
        self.conv = conv_fn(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.activation = nn.LeakyReLU(0.2)
        self.norm = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return self.activation(self.norm(self.conv(x)))

In [17]:
class UnetGenerator(nn.Module):
    def __init__(self, inshape, c_dim, nb_features=None, in_channels=3):
        super().__init__()
        self.c_dim = c_dim
        ndims = len(inshape)
        assert ndims in [1, 2, 3]
        if nb_features is None:
            nb_features = default_unet_features()
        self.enc_nf, self.dec_nf = nb_features

        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        prev_nf = in_channels + c_dim  # 3 image channels + 2 condition channels

        # Downsampling path
        self.downarm = nn.ModuleList()
        for nf in self.enc_nf:
            self.downarm.append(ConvBlock(ndims, prev_nf, nf, stride=2))
            prev_nf = nf
        self.nf_encoder = prev_nf

        # Upsampling path
        enc_history = list(reversed(self.enc_nf))
        self.uparm = nn.ModuleList()
        for i, nf in enumerate(self.dec_nf[:len(self.enc_nf)]):
            in_nf = prev_nf + enc_history[i] if i > 0 else prev_nf
            self.uparm.append(ConvBlock(ndims, in_nf, nf, stride=1))
            prev_nf = nf
        self.decoder_nf = prev_nf

        # Extra convolutions
        prev_nf += in_channels + c_dim
        self.extras = nn.ModuleList()
        for nf in self.dec_nf[len(self.enc_nf):]:
            self.extras.append(ConvBlock(ndims, prev_nf, nf, stride=1))
            prev_nf = nf

        # Final output layer
        self.out_conv = nn.Conv2d(prev_nf, in_channels, kernel_size=1)

    def forward(self, x):
        x_enc = [x]
        for layer in self.downarm:
            x_enc.append(layer(x_enc[-1]))

        x = x_enc.pop()
        for layer in self.uparm:
            x = layer(x)
            x = self.upsample(x)
            x = torch.cat([x, x_enc.pop()], dim=1)

        for layer in self.extras:
            x = layer(x)

        out = self.out_conv(x)
        return out

In [19]:
inshape = (130, 130)
generator = UnetGenerator(inshape=inshape, c_dim=2, in_channels=3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = generator.to(device)

x_input = x_input.to(device)

# Run generator
with torch.no_grad():
    x_output = generator(x_input)

print("Output shape:", x_output.shape)

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 18 but got size 17 for tensor number 1 in the list.